In [ ]:
# Mutual Fund Portfolio Mapping Coverage

This notebook extracts equity holdings from every workbook in `data/raw/Funds Monthly Portfolio Disclosure(18 funds)`, joins them to the ISIN mapping outputs, and measures mapped versus unmapped coverage for each fund.

Coverage is reported in two ways:
- **Stock-count coverage:** number of distinct holdings.
- **Portfolio-weight coverage:** sum of each holding's `% to NAV`.

The second measure is the decision signal: a small number of unmapped stocks can still represent a material part of a fund.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
FUNDS_DIR = RAW_DIR / "Funds Monthly Portfolio Disclosure(18 funds)"
MAPPED_PATH = RAW_DIR / "mapped.csv"
UNMAPPED_PATH = RAW_DIR / "unmapped.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Project root: {ROOT}")
print(f"Fund workbooks: {len(list(FUNDS_DIR.glob('*.xlsx')))}")

Project root: c:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence
Fund workbooks: 18


In [2]:
def clean_isin(value):
    """Return a normalized ISIN, or None for blanks/non-ISIN values."""
    if pd.isna(value):
        return None
    value = str(value).strip().upper()
    return value if re.fullmatch(r"IN[A-Z0-9]{10}", value) else None

mapped = pd.read_csv(MAPPED_PATH, dtype=str)
unmapped = pd.read_csv(UNMAPPED_PATH, dtype=str)

mapped["isin"] = mapped["isin"].map(clean_isin)
unmapped["isin"] = unmapped["isin"].map(clean_isin)

# One ISIN can occur more than once in the mapping output. Keep one label per ISIN.
mapped_isins = set(mapped["isin"].dropna())
unmapped_isins = set(unmapped["isin"].dropna())

print(f"Mapped ISINs: {len(mapped_isins):,}")
print(f"Unmapped ISINs: {len(unmapped_isins):,}")
print(f"Overlap between mapping files: {len(mapped_isins & unmapped_isins):,}")

Mapped ISINs: 486
Unmapped ISINs: 837
Overlap between mapping files: 0


In [10]:
def parse_number(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    text = str(value).replace(",", "").replace("%", "").strip()
    if text.startswith("(") and text.endswith(")"):
        text = "-" + text[1:-1]
    return pd.to_numeric(text, errors="coerce")


def find_column(header, keywords):
    for index, value in enumerate(header):
        label = re.sub(r"\s+", " ", str(value).strip().lower())
        if any(keyword in label for keyword in keywords):
            return index
    return None


def extract_sheet_holdings(path, sheet_name):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, dtype=object)
    raw = raw.dropna(axis=1, how="all")
    if raw.empty:
        return pd.DataFrame()

    header_idx = next(
        (
            row_index
            for row_index, row in raw.iterrows()
            if any("isin" in str(value).lower() for value in row)
        ),
        None,
    )
    if header_idx is None:
        return pd.DataFrame()

    header = raw.loc[header_idx].tolist()
    isin_col = find_column(header, ["isin"])
    value_col = find_column(header, ["market value", "fair value", "exposure"])
    weight_col = find_column(header, ["% to nav", "% to net assets", "% to net", "% to aum", "% to assets"])
    quantity_col = find_column(header, ["quantity"])
    name_col = find_column(header, ["company", "issuer", "instrument name", "name of the instrument"])

    if isin_col is None:
        return pd.DataFrame()

    records = []
    for _, row in raw.iloc[header_idx + 1 :].iterrows():
        isin = clean_isin(row.iloc[isin_col] if isin_col < len(row) else None)
        if isin is None:
            continue
        records.append(
            {
                "isin": isin,
                "holding_name": str(row.iloc[name_col]).strip() if name_col is not None and not pd.isna(row.iloc[name_col]) else None,
                "quantity": parse_number(row.iloc[quantity_col]) if quantity_col is not None else np.nan,
                "market_value_lakh": parse_number(row.iloc[value_col]) if value_col is not None else np.nan,
                "nav_weight_pct": parse_number(row.iloc[weight_col]) if weight_col is not None else np.nan,
                "source_sheet": sheet_name,
            }
        )

    return pd.DataFrame(records)


def extract_fund_workbook(path):
    fund_name = re.sub(r"^IN[A-Z0-9]{10}_", "", path.stem).strip()
    frames = [extract_sheet_holdings(path, sheet_name) for sheet_name in pd.ExcelFile(path).sheet_names]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        return pd.DataFrame()
    holdings = pd.concat(frames, ignore_index=True)
    holdings.insert(0, "fund_name", fund_name)
    holdings.insert(1, "workbook", path.name)
    return holdings


workbooks = sorted(FUNDS_DIR.glob("*.xlsx"))
fund_frames = [extract_fund_workbook(path) for path in workbooks]
fund_frames = [frame for frame in fund_frames if not frame.empty]
holdings = pd.concat(fund_frames, ignore_index=True)

# A holding should be counted once per fund even if it appears on more than one report sheet.
holdings = holdings.drop_duplicates(subset=["fund_name", "isin"], keep="first").reset_index(drop=True)
holdings["mapping_status"] = np.where(
    holdings["isin"].isin(mapped_isins), "mapped",
    np.where(holdings["isin"].isin(unmapped_isins), "unmapped", "not_in_mapping_files"),
)

print(f"Extracted {len(holdings):,} distinct fund holdings across {holdings['fund_name'].nunique():,} funds.")
display(holdings.head())

Extracted 2,484 distinct fund holdings across 18 funds.


,fund_name,workbook,isin,holding_name,quantity,market_value_lakh,nav_weight_pct,source_sheet,mapping_status
0,ICICI Prudential Balanced Advantage Fund,INF109K012B0_ICICI Prudential Balanced Advanta...,INE494B01023,TVS Motor Company Ltd.,"10,710,118.0000","464,958.3500",0.0617,BAF,mapped
1,ICICI Prudential Balanced Advantage Fund,INF109K012B0_ICICI Prudential Balanced Advanta...,INE090A01021,ICICI Bank Ltd.,"21,104,705.0000","306,862.4100",0.0407,BAF,mapped
2,ICICI Prudential Balanced Advantage Fund,INF109K012B0_ICICI Prudential Balanced Advanta...,INE040A01034,HDFC Bank Ltd.,"34,591,493.0000","245,253.6900",0.0325,BAF,mapped
3,ICICI Prudential Balanced Advantage Fund,INF109K012B0_ICICI Prudential Balanced Advanta...,INE002A01018,Reliance Industries Ltd.,"15,358,623.0000","196,129.6200",0.0260,BAF,mapped
4,ICICI Prudential Balanced Advantage Fund,INF109K012B0_ICICI Prudential Balanced Advanta...,INE009A01021,Infosys Ltd.,"16,393,365.0000","185,867.9700",0.0247,BAF,mapped


In [11]:
# Some disclosures store weights as decimals (0.0617), while others store percentages (4.28).
# Normalize everything to percentage points before aggregating.
fund_max_weight = holdings.groupby("fund_name")["nav_weight_pct"].transform("max")
decimal_weight_mask = fund_max_weight.le(1.5)
holdings.loc[decimal_weight_mask, "nav_weight_pct"] = (
    holdings.loc[decimal_weight_mask, "nav_weight_pct"] * 100
)
holdings["coverage_scope"] = holdings["mapping_status"].isin(["mapped", "unmapped"])

coverage = (
    holdings[holdings["coverage_scope"]]
    .groupby(["fund_name", "mapping_status"], as_index=False)
    .agg(
        stock_count=("isin", "nunique"),
        portfolio_weight_pct=("nav_weight_pct", "sum"),
        market_value_lakh=("market_value_lakh", "sum"),
    )
)

fund_totals = (
    coverage.groupby("fund_name", as_index=False)
    .agg(
        total_stocks=("stock_count", "sum"),
        total_portfolio_weight_pct=("portfolio_weight_pct", "sum"),
        total_market_value_lakh=("market_value_lakh", "sum"),
    )
)

coverage = coverage.merge(fund_totals, on="fund_name", how="left")
coverage["stock_count_coverage_pct"] = 100 * coverage["stock_count"] / coverage["total_stocks"]
coverage["portfolio_weight_coverage_pct"] = 100 * coverage["portfolio_weight_pct"] / coverage["total_portfolio_weight_pct"]

coverage_summary = (
    coverage.pivot(index="fund_name", columns="mapping_status", values=[
        "stock_count", "portfolio_weight_pct", "stock_count_coverage_pct", "portfolio_weight_coverage_pct"
    ])
    .fillna(0)
)
coverage_summary.columns = ["_".join(column).strip() for column in coverage_summary.columns]
coverage_summary = coverage_summary.reset_index()

unknown_summary = (
    holdings[~holdings["coverage_scope"]]
    .groupby("fund_name", as_index=False)
    .agg(not_in_mapping_stock_count=("isin", "nunique"), not_in_mapping_weight_pct=("nav_weight_pct", "sum"))
)

coverage_summary = coverage_summary.merge(unknown_summary, on="fund_name", how="left").fillna(0)
coverage_summary = coverage_summary.sort_values("portfolio_weight_coverage_pct_unmapped", ascending=False)

display(coverage_summary)

,fund_name,stock_count_mapped,stock_count_unmapped,portfolio_weight_pct_mapped,portfolio_weight_pct_unmapped,stock_count_coverage_pct_mapped,stock_count_coverage_pct_unmapped,portfolio_weight_coverage_pct_mapped,portfolio_weight_coverage_pct_unmapped,not_in_mapping_stock_count,not_in_mapping_weight_pct
0,Aditya Birla Sun Life Liquid Fund,0.0000,197.0000,0.0000,83.1335,0.0000,100.0000,0.0000,100.0000,17.0000,16.0968
2,HDFC Corporate Bond Fund,0.0000,182.0000,0.0000,79.6300,0.0000,100.0000,0.0000,100.0000,56.0000,16.1500
7,ICICI Prudential Corporate Bond Fund,0.0000,152.0000,0.0000,77.3223,0.0000,100.0000,0.0000,100.0000,35.0000,19.7400
4,HDFC Liquid Fund,0.0000,138.0000,0.0000,81.8800,0.0000,100.0000,0.0000,100.0000,13.0000,16.8800
15,SBI LIQUID FUND,0.0000,142.0000,0.0000,79.1300,0.0000,100.0000,0.0000,100.0000,13.0000,16.8900
1,HDFC Balanced Advantage Fund,165.0000,117.0000,73.2600,18.3100,58.5106,41.4894,80.0044,19.9956,42.0000,7.0800
6,ICICI Prudential Balanced Advantage Fund,101.0000,63.0000,70.3101,14.4507,61.5854,38.4146,82.9512,17.0488,47.0000,7.7175
13,Parag Parikh Flexi Cap Fund,59.0000,47.0000,73.0800,11.6300,55.6604,44.3396,86.2708,13.7292,4.0000,0.8300
3,HDFC Flexi Cap Fund,78.0000,2.0000,94.8400,2.1900,97.5000,2.5000,97.7430,2.2570,3.0000,0.4500
14,SBI CONTRA FUND,83.0000,1.0000,86.7300,0.9700,98.8095,1.1905,98.8940,1.1060,5.0000,2.7400


In [13]:
# Change these thresholds to match the business rule you want to use.
LOW_UNMAPPED_STOCK_COUNT_PCT = 5.0
MATERIAL_UNMAPPED_WEIGHT_PCT = 1.0

review_queue = coverage_summary[
    [
        "fund_name",
        "stock_count_coverage_pct_unmapped",
        "portfolio_weight_coverage_pct_unmapped",
        "stock_count_unmapped",
        "portfolio_weight_pct_unmapped",
        "not_in_mapping_stock_count",
        "not_in_mapping_weight_pct",
    ]
].copy()

# Debt and liquid funds are intentionally reported but should not be judged as stock mapping coverage.
debt_or_liquid_mask = review_queue["fund_name"].str.contains(
    "liquid|corporate bond", case=False, regex=True
)
review_queue["review_decision"] = np.select(
    [
        debt_or_liquid_mask,
        review_queue["portfolio_weight_coverage_pct_unmapped"] >= MATERIAL_UNMAPPED_WEIGHT_PCT,
        review_queue["stock_count_coverage_pct_unmapped"] <= LOW_UNMAPPED_STOCK_COUNT_PCT,
    ],
    [
        "NOT STOCK COVERAGE: debt/liquid fund",
        "REVIEW: material unmapped portfolio weight",
        "LOW PRIORITY: low unmapped stock-count coverage",
    ],
    default="REVIEW: unmapped holdings need assessment",
)

review_queue = review_queue.sort_values(
    ["review_decision", "portfolio_weight_coverage_pct_unmapped"], ascending=[True, False]
)
display(review_queue)

,fund_name,stock_count_coverage_pct_unmapped,portfolio_weight_coverage_pct_unmapped,stock_count_unmapped,portfolio_weight_pct_unmapped,not_in_mapping_stock_count,not_in_mapping_weight_pct,review_decision
5,HDFC Retirement Fund - Equity Plan,2.8169,0.6904,2.0000,0.6500,1.0000,0.9400,LOW PRIORITY: low unmapped stock-count coverage
8,ICICI Prudential Large Cap Fund,3.4483,0.5200,3.0000,0.4980,3.0000,0.7629,LOW PRIORITY: low unmapped stock-count coverage
9,Kotak Large & Mid Cap Fund,1.8182,0.0233,1.0000,0.0200,13.0000,12.1400,LOW PRIORITY: low unmapped stock-count coverage
12,Nippon India Small Cap Fund,0.3937,0.0103,1.0000,0.0100,0.0000,0.0000,LOW PRIORITY: low unmapped stock-count coverage
10,Nippon India Large Cap Fund,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,LOW PRIORITY: low unmapped stock-count coverage
11,Nippon India Retirement Fund- Wealth Creation ...,1.4085,0.0000,1.0000,0.0000,0.0000,0.0000,LOW PRIORITY: low unmapped stock-count coverage
16,SBI Nifty 50 ETF,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,LOW PRIORITY: low unmapped stock-count coverage
17,UTI Nifty 50 Index Fund,0.0000,0.0000,0.0000,0.0000,1.0000,0.0200,LOW PRIORITY: low unmapped stock-count coverage
0,Aditya Birla Sun Life Liquid Fund,100.0000,100.0000,197.0000,83.1335,17.0000,16.0968,NOT STOCK COVERAGE: debt/liquid fund
2,HDFC Corporate Bond Fund,100.0000,100.0000,182.0000,79.6300,56.0000,16.1500,NOT STOCK COVERAGE: debt/liquid fund


In [14]:
# Inspect the exact unmapped positions that drive each fund's review decision.
unmapped_holdings = (
    holdings[holdings["mapping_status"] == "unmapped"]
    .loc[:, ["fund_name", "holding_name", "isin", "quantity", "market_value_lakh", "nav_weight_pct", "source_sheet"]]
    .sort_values(["fund_name", "nav_weight_pct"], ascending=[True, False])
)

display(unmapped_holdings)

# Save reusable outputs beside the raw mapping files.
OUTPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)
coverage_summary.to_csv(OUTPUT_DIR / "fund_mapping_coverage_summary.csv", index=False)
review_queue.to_csv(OUTPUT_DIR / "fund_mapping_review_queue.csv", index=False)
holdings.to_csv(OUTPUT_DIR / "fund_holdings_mapping_status.csv", index=False)

print(f"Saved analysis outputs to {OUTPUT_DIR}")

,fund_name,holding_name,isin,quantity,market_value_lakh,nav_weight_pct,source_sheet
2249,Aditya Birla Sun Life Liquid Fund,Union Bank of India (01/10/2026) **#,INE692A16MR3,"39,000.0000","193,999.4600",2.7782,CASH
2250,Aditya Birla Sun Life Liquid Fund,HDFC Bank Ltd. (11/09/2026) **#,INE040A16HN4,"31,000.0000","154,741.4600",2.2160,CASH
2251,Aditya Birla Sun Life Liquid Fund,City Union Bank Ltd. (10/09/2026) **#,INE491A16169,"24,000.0000","119,803.5600",1.7156,CASH
2252,Aditya Birla Sun Life Liquid Fund,HDFC Bank Ltd. (06/11/2026) **#,INE040A16HU9,"23,000.0000","113,676.4700",1.6279,CASH
2120,Aditya Birla Sun Life Liquid Fund,National Bank for Agriculture and Rural Develo...,INE261F14PT0,"20,000.0000","99,830.7000",1.4296,CASH
...,...,...,...,...,...,...,...
1618,SBI LIQUID FUND,Godrej Properties Ltd.,INE484J14F63,"2,000.0000","9,868.7000",0.1100,SLF
1517,SBI LIQUID FUND,REC Ltd.,INE020B08FH7,"5,000.0000","5,004.3900",0.0500,SLF
1619,SBI LIQUID FUND,National Bank for Agriculture and Rural Develo...,INE261F14QA8,"1,000.0000","4,938.2400",0.0500,SLF
1620,SBI LIQUID FUND,Astec Lifesciences Ltd.,INE563J14DP5,500.0000,"2,492.2600",0.0300,SLF


Saved analysis outputs to c:\Users\313635\Downloads\Mutual Fund Disclosure Intelligence\data\processed
